In [1]:
# Day 3 - Step 1: Load Pair Dataset

import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

# --- Absolute paths (corrected for your system) ---
PAIR_CSV = r"D:\PEC\HSI_Project\processed\pair_index.csv"
TENSOR_DIR = r"D:\PEC\HSI_Project\tensors\sentences"

# --- Dataset Class ---
class HSIPairDataset(Dataset):
    def __init__(self, pair_csv, tensor_dir):
        self.df = pd.read_csv(pair_csv)
        self.tensor_dir = tensor_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        tensor_A = torch.load(os.path.join(self.tensor_dir, row['tensor_A']))
        tensor_B = torch.load(os.path.join(self.tensor_dir, row['tensor_B']))
        same_writer = torch.tensor(row['same_writer'], dtype=torch.float32)
        same_pen = torch.tensor(row['same_pen'], dtype=torch.float32)
        return tensor_A, tensor_B, same_writer, same_pen

# --- Instantiate dataset ---
dataset = HSIPairDataset(PAIR_CSV, TENSOR_DIR)

# --- Check dataset size ---
print("Total pairs:", len(dataset))

# --- Test DataLoader ---
loader = DataLoader(dataset, batch_size=8, shuffle=True)

# --- Fetch one batch ---
tensor_A, tensor_B, same_writer, same_pen = next(iter(loader))
print("Tensor A shape:", tensor_A.shape)
print("Tensor B shape:", tensor_B.shape)
print("Writer labels:", same_writer)
print("Pen labels:", same_pen)


Total pairs: 10000


C:\Users\dell\AppData\Local\Temp\ipykernel_13028\3203930563.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  tensor_A = torch.load(os.path.join(self.tensor_dir, row['ten

Tensor A shape: torch.Size([8, 149, 78, 515])
Tensor B shape: torch.Size([8, 149, 78, 515])
Writer labels: tensor([1., 0., 0., 1., 0., 0., 0., 0.])
Pen labels: tensor([0., 0., 0., 0., 0., 0., 0., 1.])


In [2]:
import torch
import torch.nn as nn

class HSI_CNN_LSTM(nn.Module):
    def __init__(self, input_height=78, input_width=515, num_bands=149, lstm_hidden=128, num_classes=2):
        super(HSI_CNN_LSTM, self).__init__()
        
        # CNN: Spatial feature extraction
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),  # input: 1 channel per band
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        # Compute CNN output size dynamically
        self._cnn_output_size = self._get_cnn_output_size(input_height, input_width)
        
        # LSTM: Sequential spectral feature learning
        self.lstm = nn.LSTM(input_size=self._cnn_output_size, hidden_size=lstm_hidden, batch_first=True)
        
        # Fully connected layers for verification
        self.fc = nn.Sequential(
            nn.Linear(lstm_hidden, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)  # Output: 2 (same/different)
        )

    def _get_cnn_output_size(self, h, w):
        # Dummy input to compute flattened size
        x = torch.zeros(1, 1, h, w)
        x = self.cnn(x)
        return x.view(1, -1).size(1)
    
    def forward(self, x):
        # x: [B, num_bands, H, W]
        B, bands, H, W = x.size()
        cnn_out = []
        for b in range(bands):
            band_feat = self.cnn(x[:, b:b+1, :, :])  # [B, C, H', W']
            band_feat = band_feat.view(B, -1)
            cnn_out.append(band_feat)
        
        seq_input = torch.stack(cnn_out, dim=1)  # [B, bands, cnn_out_size]
        lstm_out, _ = self.lstm(seq_input)
        final_feat = lstm_out[:, -1, :]  # last timestep
        out = self.fc(final_feat)
        return out


In [3]:
# Example initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HSI_CNN_LSTM(input_height=78, input_width=515, num_bands=149).to(device)

print("Model initialized on", device)


Model initialized on cuda


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class HSICNN(nn.Module):
    def __init__(self, embedding_dim=128):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(149, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.fc = nn.Linear(64, embedding_dim)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


In [5]:
class PairClassifier(nn.Module):
    def __init__(self, embed_dim=128):
        super().__init__()
        self.fc = nn.Linear(embed_dim * 2, 1)

    def forward(self, f1, f2):
        x = torch.cat([f1, f2], dim=1)
        return self.fc(x).squeeze(1)


In [6]:
device = "cuda"

backbone = HSICNN().to(device)
head = PairClassifier().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(
    list(backbone.parameters()) + list(head.parameters()),
    lr=1e-4
)


In [7]:
def pad_height(x, target=78):
    h = x.shape[1]
    if h < target:
        pad = target - h
        x = F.pad(x, (0, 0, 0, pad))
    return x


In [8]:
from torch.utils.data import DataLoader

loader = DataLoader(dataset, batch_size=2, shuffle=True)

backbone.train()
head.train()

for step, (A, B, y, _) in enumerate(loader):

    A = A.float().to(device)
    B = B.float().to(device)
    y = y.float().to(device)

    A = pad_height(A)
    B = pad_height(B)

    optimizer.zero_grad()

    f1 = backbone(A)
    f2 = backbone(B)

    logits = head(f1, f2)
    loss = criterion(logits, y)

    loss.backward()
    optimizer.step()

    if step % 50 == 0:
        print(f"Step {step} | Loss: {loss.item():.4f}")

    if step == 500:
        break


C:\Users\dell\AppData\Local\Temp\ipykernel_13968\3203930563.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  tensor_A = torch.load(os.path.join(self.tensor_dir, row['ten

Step 0 | Loss: 0.6946
Step 50 | Loss: 1.0805
Step 100 | Loss: 0.3746
Step 150 | Loss: 0.3579
Step 200 | Loss: 0.3766
Step 250 | Loss: 0.3940
Step 300 | Loss: 0.7938
Step 350 | Loss: 0.8108
Step 400 | Loss: 0.2792
Step 450 | Loss: 0.8536
Step 500 | Loss: 0.7929


In [5]:
# ============================
# Day 3 | Step 3: Training
# ============================

import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# -----------------------------
# Dataset
# -----------------------------
class HSIPairDataset(Dataset):
    def __init__(self, pair_csv, tensor_dir):
        import pandas as pd
        self.df = pd.read_csv(pair_csv)
        self.tensor_dir = tensor_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path_A = os.path.join(self.tensor_dir, row['tensor_A'])
        path_B = os.path.join(self.tensor_dir, row['tensor_B'])

        tensor_A = torch.load(path_A, weights_only=True)
        tensor_B = torch.load(path_B, weights_only=True)

        # Ensure shape [bands, H, W]
        if tensor_A.dim() != 3 or tensor_B.dim() != 3:
            raise ValueError("Tensors must have shape [bands, H, W]")

        same_writer = torch.tensor(row['same_writer'], dtype=torch.float32)
        same_pen = torch.tensor(row['same_pen'], dtype=torch.float32)

        return tensor_A, tensor_B, same_writer, same_pen


# -----------------------------
# Contrastive Loss
# -----------------------------
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin
        self.eps = 1e-9

    def forward(self, output1, output2, label):
        # label = 1 for same writer, 0 for different
        euclidean_distance = torch.sqrt(torch.sum((output1 - output2)**2, dim=1) + self.eps)
        loss = label * euclidean_distance**2 + (1 - label) * torch.clamp(self.margin - euclidean_distance, min=0.0)**2
        return torch.mean(loss)


# -----------------------------
# Example HSI CNN-LSTM model
# -----------------------------
class HSI_CNN_LSTM(nn.Module):
    def __init__(self, bands=149, lstm_hidden=128, lstm_layers=1):
        super(HSI_CNN_LSTM, self).__init__()
        # CNN for band-level feature extraction
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        # LSTM for sequence of bands
        self.lstm = nn.LSTM(32, lstm_hidden, lstm_layers, batch_first=True)
        self.fc = nn.Linear(lstm_hidden, 64)  # Output embedding

    def forward(self, x):
        # x shape: [B, bands, H, W]
        B, bands, H, W = x.shape
        cnn_out = []
        for b in range(bands):
            band_feat = self.cnn(x[:, b:b+1, :, :])  # [B, C, 1, 1]
            band_feat = band_feat.view(B, -1)        # [B, C]
            cnn_out.append(band_feat)
        seq = torch.stack(cnn_out, dim=1)  # [B, bands, C]
        lstm_out, _ = self.lstm(seq)       # [B, bands, hidden]
        lstm_last = lstm_out[:, -1, :]     # Last time step
        out = self.fc(lstm_last)           # [B, embedding]
        return out


# -----------------------------
# Paths
# -----------------------------
PAIR_CSV = r"D:\PEC\HSI_Project\processed\pair_index.csv"
TENSOR_DIR = r"D:\PEC\HSI_Project\tensors\sentences"

# -----------------------------
# Hyperparameters
# -----------------------------
BATCH_SIZE = 2   # Use small batch to avoid OOM
NUM_EPOCHS = 5
LR = 1e-4

# -----------------------------
# Dataset & Loader
# -----------------------------
dataset = HSIPairDataset(PAIR_CSV, TENSOR_DIR)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# -----------------------------
# Model, loss, optimizer
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HSI_CNN_LSTM().to(device)
criterion = ContrastiveLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# Mixed precision
scaler = torch.amp.GradScaler()  # updated new API

# -----------------------------
# Training loop
# -----------------------------
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for step, (tensor_A, tensor_B, same_writer, _) in enumerate(loader):
        tensor_A = tensor_A.float().to(device)
        tensor_B = tensor_B.float().to(device)
        same_writer = same_writer.to(device)

        optimizer.zero_grad()
        with torch.amp.autocast(device_type=device):
            output1 = model(tensor_A)
            output2 = model(tensor_B)
            loss = criterion(output1, output2, same_writer)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        if step % 50 == 0:
            print(f"Step {step} | Loss: {loss.item():.4f}")

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] - Avg Loss: {total_loss/len(loader):.4f}")


Step 0 | Loss: 0.9966
Step 50 | Loss: 0.9820
Step 100 | Loss: 0.5844
Step 150 | Loss: 0.0182
Step 200 | Loss: 0.9923
Step 250 | Loss: 0.4228
Step 300 | Loss: 0.5712
Step 350 | Loss: 0.1592
Step 400 | Loss: 0.4999
Step 450 | Loss: 0.5328
Step 500 | Loss: 0.0177
Step 550 | Loss: 0.1299
Step 600 | Loss: 0.4140
Step 650 | Loss: 0.0740
Step 700 | Loss: 0.6498
Step 750 | Loss: 0.7506
Step 800 | Loss: 0.8464
Step 850 | Loss: 0.0630
Step 900 | Loss: 0.3828
Step 950 | Loss: 0.6969
Step 1000 | Loss: 0.5789
Step 1050 | Loss: 0.5891
Step 1100 | Loss: 0.5108
Step 1150 | Loss: 0.4573
Step 1200 | Loss: 0.0083
Step 1250 | Loss: 0.1945
Step 1300 | Loss: 0.6855
Step 1350 | Loss: 0.4918
Step 1400 | Loss: 0.1603
Step 1450 | Loss: 0.4725
Step 1500 | Loss: 0.1947
Step 1550 | Loss: 0.3249
Step 1600 | Loss: 0.0137
Step 1650 | Loss: 0.5644
Step 1700 | Loss: 0.0773
Step 1750 | Loss: 0.0709
Step 1800 | Loss: 0.4707
Step 1850 | Loss: 0.1149
Step 1900 | Loss: 0.8597
Step 1950 | Loss: 0.0460
Step 2000 | Loss: 0.537

In [4]:
# ============================
# Day 3 | Step 3: Training with Early Stopping
# ============================

import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# -----------------------------
# Dataset
# -----------------------------
class HSIPairDataset(Dataset):
    def __init__(self, pair_csv, tensor_dir):
        import pandas as pd
        self.df = pd.read_csv(pair_csv)
        self.tensor_dir = tensor_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path_A = os.path.join(self.tensor_dir, row['tensor_A'])
        path_B = os.path.join(self.tensor_dir, row['tensor_B'])

        tensor_A = torch.load(path_A, weights_only=True)
        tensor_B = torch.load(path_B, weights_only=True)

        if tensor_A.dim() != 3 or tensor_B.dim() != 3:
            raise ValueError("Expected tensor shape [bands, H, W]")

        same_writer = torch.tensor(row['same_writer'], dtype=torch.float32)
        return tensor_A, tensor_B, same_writer

# -----------------------------
# Contrastive Loss
# -----------------------------
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin
        self.eps = 1e-9

    def forward(self, out1, out2, label):
        dist = torch.sqrt(torch.sum((out1 - out2) ** 2, dim=1) + self.eps)
        loss = label * dist**2 + (1 - label) * torch.clamp(self.margin - dist, min=0.0) ** 2
        return loss.mean()

# -----------------------------
# HSI CNN-LSTM Model
# -----------------------------
class HSI_CNN_LSTM(nn.Module):
    def __init__(self, bands=149, lstm_hidden=128):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.lstm = nn.LSTM(32, lstm_hidden, batch_first=True)
        self.fc = nn.Linear(lstm_hidden, 64)

    def forward(self, x):
        B, bands, H, W = x.shape
        features = []
        for b in range(bands):
            f = self.cnn(x[:, b:b+1])
            features.append(f.view(B, -1))
        seq = torch.stack(features, dim=1)
        lstm_out, _ = self.lstm(seq)
        return self.fc(lstm_out[:, -1])

# -----------------------------
# Paths & Hyperparameters
# -----------------------------
PAIR_CSV = r"D:\PEC\HSI_Project\processed\pair_index.csv"
TENSOR_DIR = r"D:\PEC\HSI_Project\tensors\sentences"

BATCH_SIZE = 2
NUM_EPOCHS = 10
LR = 1e-4

# -----------------------------
# Dataset & Loader
# -----------------------------
dataset = HSIPairDataset(PAIR_CSV, TENSOR_DIR)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# -----------------------------
# Model, Loss, Optimizer
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HSI_CNN_LSTM().to(device)
criterion = ContrastiveLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scaler = torch.amp.GradScaler()

# -----------------------------
# Early Stopping Setup
# -----------------------------
best_loss = float("inf")
patience = 3            # stop if no improvement for 3 consecutive epochs
patience_counter = 0

# -----------------------------
# Store losses
# -----------------------------
epoch_losses = []

# -----------------------------
# Training Loop
# -----------------------------
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0

    for step, (A, B, label) in enumerate(loader):
        A = A.float().to(device)
        B = B.float().to(device)
        label = label.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast(device_type=device):
            out1 = model(A)
            out2 = model(B)
            loss = criterion(out1, out2, label)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        if step % 50 == 0:
            print(f"Step {step} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(loader)
    epoch_losses.append(avg_loss)
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] - Avg Loss: {avg_loss:.4f}")

    # -----------------------------
    # Early Stopping Logic
    # -----------------------------
    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
        torch.save(model.state_dict(), "best_model_day3_step3.pt")
        print("✅ Best model updated")
    else:
        patience_counter += 1
        print(f"⚠ No improvement. Patience: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print("🛑 Early stopping triggered!")
            break

print("🎉 Training completed")


Step 0 | Loss: 0.9927
Step 50 | Loss: 0.9773
Step 100 | Loss: 0.9421
Step 150 | Loss: 0.3328
Step 200 | Loss: 0.3142
Step 250 | Loss: 0.0880
Step 300 | Loss: 0.5436
Step 350 | Loss: 0.0809
Step 400 | Loss: 0.0399
Step 450 | Loss: 0.4898
Step 500 | Loss: 0.1954
Step 550 | Loss: 0.3385
Step 600 | Loss: 0.3192
Step 650 | Loss: 0.8112
Step 700 | Loss: 0.3820
Step 750 | Loss: 0.1250
Step 800 | Loss: 0.0607
Step 850 | Loss: 0.2191
Step 900 | Loss: 0.0114
Step 950 | Loss: 0.3018
Step 1000 | Loss: 0.5195
Step 1050 | Loss: 0.7525
Step 1100 | Loss: 0.1488
Step 1150 | Loss: 0.0305
Step 1200 | Loss: 0.5408
Step 1250 | Loss: 0.4150
Step 1300 | Loss: 2.9033
Step 1350 | Loss: 0.3540
Step 1400 | Loss: 0.0008
Step 1450 | Loss: 0.1215
Step 1500 | Loss: 0.8956
Step 1550 | Loss: 0.5027
Step 1600 | Loss: 0.0214
Step 1650 | Loss: 0.4763
Step 1700 | Loss: 0.4273
Step 1750 | Loss: 0.4585
Step 1800 | Loss: 0.3294
Step 1850 | Loss: 0.3688
Step 1900 | Loss: 0.4616
Step 1950 | Loss: 0.3734
Step 2000 | Loss: 0.020

In [5]:
import os
import json
from datetime import datetime
import torch

# -----------------------------
# Paths for saving
# -----------------------------
SAVE_DIR = r"D:\PEC\HSI_Project\checkpoints\day3_step3"
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_PATH = os.path.join(SAVE_DIR, "hsi_cnn_lstm_contrastive.pt")
OPT_PATH   = os.path.join(SAVE_DIR, "optimizer.pt")
LOSS_PATH  = os.path.join(SAVE_DIR, "epoch_losses.json")
META_PATH  = os.path.join(SAVE_DIR, "training_meta.json")

# -----------------------------
# 1. Save model weights
# -----------------------------
torch.save(model.state_dict(), MODEL_PATH)

# -----------------------------
# 2. Save optimizer state
# -----------------------------
torch.save(optimizer.state_dict(), OPT_PATH)

# -----------------------------
# 3. Save epoch-wise losses
# -----------------------------
with open(LOSS_PATH, "w") as f:
    json.dump(epoch_losses, f, indent=4)

# -----------------------------
# 4. Save training metadata
# -----------------------------
meta = {
    "model": "HSI_CNN_LSTM (Contrastive Learning)",
    "task": "Day 3 - Step 3",
    "epochs_ran": len(epoch_losses),
    "learning_rate": optimizer.param_groups[0]["lr"],
    "batch_size": BATCH_SIZE,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

with open(META_PATH, "w") as f:
    json.dump(meta, f, indent=4)

print("✅ Day 3 Step 3 saved successfully!")
print(f"📦 Model      : {MODEL_PATH}")
print(f"📦 Optimizer  : {OPT_PATH}")
print(f"📊 Losses     : {LOSS_PATH}")
print(f"📝 Metadata   : {META_PATH}")



✅ Day 3 Step 3 saved successfully!
📦 Model      : D:\PEC\HSI_Project\checkpoints\day3_step3\hsi_cnn_lstm_contrastive.pt
📦 Optimizer  : D:\PEC\HSI_Project\checkpoints\day3_step3\optimizer.pt
📊 Losses     : D:\PEC\HSI_Project\checkpoints\day3_step3\epoch_losses.json
📝 Metadata   : D:\PEC\HSI_Project\checkpoints\day3_step3\training_meta.json
